In [1]:
import requests
import pandas as pd
import numpy as np
from collections import Counter

In [2]:
# Load AFINN lexicon
afinn_url = "https://raw.githubusercontent.com/fnielsen/afinn/master/afinn/data/AFINN-en-165.txt"
afinn_df = pd.read_csv(afinn_url, sep='\t', header=None, names=['word', 'score'])

# Ensure the 'score' column is numeric
afinn_df['score'] = pd.to_numeric(afinn_df['score'], errors='coerce')

In [3]:
# Load VADER lexicon
vader_url = "https://raw.githubusercontent.com/cjhutto/vaderSentiment/master/vaderSentiment/vader_lexicon.txt"
vader_df = pd.read_csv(vader_url, sep='\t', header=None, names=['word', 'score'])

# Ensure the 'score' column is numeric
vader_df['score'] = pd.to_numeric(vader_df['score'], errors='coerce')

In [6]:
# Load NRC Emotion Lexicon from Excel file
nrc_file_path = "E:/Works/10. Mental Health Disorder/Dataset/Extra Documents/NRC-Emotion-Lexicon-v0.92-In105Languages-Nov2017Translations.xlsx"

# Load the Excel file
xl = pd.ExcelFile(nrc_file_path)
nrc_data = xl.parse(xl.sheet_names[0])

# Process NRC Emotion Lexicon data
nrc_words = {}
for index, row in nrc_data.iterrows():
    word = row['English (en)'] 
    emotions = row[1:].to_dict() 
    if word not in nrc_words:
        nrc_words[word] = []
    for emotion, value in emotions.items():
        if value == 1:
            nrc_words[word].append(emotion)

In [7]:
from nltk.stem import PorterStemmer

# Initialize the stemmer
ps = PorterStemmer()

In [8]:
# Function to preprocess words
def preprocess_word(word):
    if isinstance(word, str):
        word = word.lower()
        word = ps.stem(word)
    return word

# Apply preprocessing
afinn_df['word'] = afinn_df['word'].apply(preprocess_word)
vader_df['word'] = vader_df['word'].apply(preprocess_word)

# Preprocess NRC words
nrc_words_processed = {}
for word, emotions in nrc_words.items():
    processed_word = preprocess_word(word)
    nrc_words_processed[processed_word] = emotions


In [9]:
def find_negative_words(text, afinn_df, vader_df, nrc_words):
    words = text.split()
    found_words = []

    # Check AFINN words
    afinn_negative = afinn_df[afinn_df['score'] < 0]['word'].tolist()
    found_words.extend([word for word in words if word in afinn_negative])

    # Check VADER words
    vader_negative = vader_df[vader_df['score'] < 0]['word'].tolist()
    found_words.extend([word for word in words if word in vader_negative])

    # Check NRC words
    negative_emotions = ['anger', 'disgust', 'fear', 'sadness']
    for word in words:
        if word in nrc_words and any(emotion in negative_emotions for emotion in nrc_words[word]):
            found_words.append(word)

    return list(set(found_words))

In [10]:
# Load the dataset
df = pd.read_csv('E:/Works/10. Mental Health Disorder/Dataset/2_types_preprocessed_3lakh.csv', encoding='utf-8')
df.head()

,text,Label,filtered_text_eng,tokenized_text,Preprocessed_Text
0,"I'm done with it all. Any tips?First of all, i...",2,done with it Any of if going to comment get or...,"['done', 'go', 'comment', 'get', 'plea', 'unde...",done go comment get plea understand want hear ...
1,i 20m was a problem child when grow up i frequ...,1,i was a problem child when grow up i frequent ...,"['problem', 'child', 'grow', 'frequent', 'got'...",problem child grow frequent got troubl junior ...
2,I officially hate my school We have to start p...,0,I officially hate my school We have to start o...,"['offici', 'hate', 'school', 'start', 'enter',...",offici hate school start enter caus fix
3,but onc again depress love to take that from m...,1,but onc again depress love to take that from m...,"['onc', 'depress', 'love', 'take', 'angri', 'w...",onc depress love take angri want cri wont come...
4,"Starting today, I'm going to attempt to fix my...",0,Starting going to attempt to fix my sleep sche...,"['start', 'go', 'attempt', 'fix', 'sleep', 'sc...",start go attempt fix sleep schedul school coup...


In [11]:
df.isnull().sum()

text                    0
Label                   0
filtered_text_eng     380
tokenized_text          0
Preprocessed_Text    1029
dtype: int64

In [12]:
df = df.dropna()

In [13]:
# Count occurrences of each label
df['Label'].value_counts()

Label
1    99985
2    99737
0    99249
Name: count, dtype: int64

In [14]:
df.shape

(298971, 5)

In [15]:
# Apply the function to each sentence and create a new column
def process_negative_words(row):
    if row['Label'] == 0:
        return row['tokenized_text']
    else:
        return find_negative_words(row['Preprocessed_Text'], afinn_df, vader_df, nrc_words_processed)

df['filtered_tokenized_words'] = df.apply(process_negative_words, axis=1)

In [16]:
df.head(20)

,text,Label,filtered_text_eng,tokenized_text,Preprocessed_Text,filtered_tokenized_words
0,"I'm done with it all. Any tips?First of all, i...",2,done with it Any of if going to comment get or...,"['done', 'go', 'comment', 'get', 'plea', 'unde...",done go comment get plea understand want hear ...,"[sorri, pain, kill, hate]"
1,i 20m was a problem child when grow up i frequ...,1,i was a problem child when grow up i frequent ...,"['problem', 'child', 'grow', 'frequent', 'got'...",problem child grow frequent got troubl junior ...,"[awkward, bulli, lost, cri, hard, wrong, troub..."
2,I officially hate my school We have to start p...,0,I officially hate my school We have to start o...,"['offici', 'hate', 'school', 'start', 'enter',...",offici hate school start enter caus fix,"['offici', 'hate', 'school', 'start', 'enter',..."
3,but onc again depress love to take that from m...,1,but onc again depress love to take that from m...,"['onc', 'depress', 'love', 'take', 'angri', 'w...",onc depress love take angri want cri wont come...,"[tire, angri, damn, cri, depress]"
4,"Starting today, I'm going to attempt to fix my...",0,Starting going to attempt to fix my sleep sche...,"['start', 'go', 'attempt', 'fix', 'sleep', 'sc...",start go attempt fix sleep schedul school coup...,"['start', 'go', 'attempt', 'fix', 'sleep', 'sc..."
5,i first start off hate peopleand then even mor...,1,i first start off hate then even more peopl an...,"['first', 'start', 'hate', 'even', 'peopl', 'c...",first start hate even peopl came conclus hate ...,"[ruin, suck, idiot, hate, hurt, fine]"
6,Honestly I don’t know what to doI feel like my...,2,Honestly I know what to feel like my future wi...,"['honest', 'know', 'feel', 'like', 'futur', 'g...",honest know feel like futur go addict alcohol ...,"[disappear, disappoint, struggl]"
7,Whoever comments garlic bread on this post Has...,0,Whoever garlic bread on this post to eat whole...,"['whoever', 'garlic', 'bread', 'post', 'eat', ...",whoever garlic bread post eat whole chees,"['whoever', 'garlic', 'bread', 'post', 'eat', ..."
8,I just don't know what to doFor about 2 years ...,2,I just know what to about had I quit school at...,"['know', 'quit', 'school', 'travel', 'got', 'l...",know quit school travel got lot short term bac...,"[difficult, hate]"
9,Could I get some help So my Spanish teach had ...,0,Could I get some help So my teach had the glor...,"['could', 'get', 'help', 'teach', 'glorious', ...",could get help teach glorious idea project cre...,"['could', 'get', 'help', 'teach', 'glorious', ..."


In [17]:
df.shape

(298971, 6)

In [18]:
# Remove rows where negative_words column is empty or zero
df = df[df['filtered_tokenized_words'].apply(len) > 0]

In [19]:
df.shape

(287398, 6)

In [20]:
# Count occurrences of each label
df['Label'].value_counts()

Label
0    99249
1    95917
2    92232
Name: count, dtype: int64

In [21]:
df.isnull().sum()

text                        0
Label                       0
filtered_text_eng           0
tokenized_text              0
Preprocessed_Text           0
filtered_tokenized_words    0
dtype: int64

In [22]:
df.to_csv('E:/Works/10. Mental Health Disorder/Dataset/2_types_filtered_features_3lakh.csv', index=False, encoding='utf-8')

In [23]:
df.head()

,text,Label,filtered_text_eng,tokenized_text,Preprocessed_Text,filtered_tokenized_words
0,"I'm done with it all. Any tips?First of all, i...",2,done with it Any of if going to comment get or...,"['done', 'go', 'comment', 'get', 'plea', 'unde...",done go comment get plea understand want hear ...,"[sorri, pain, kill, hate]"
1,i 20m was a problem child when grow up i frequ...,1,i was a problem child when grow up i frequent ...,"['problem', 'child', 'grow', 'frequent', 'got'...",problem child grow frequent got troubl junior ...,"[awkward, bulli, lost, cri, hard, wrong, troub..."
2,I officially hate my school We have to start p...,0,I officially hate my school We have to start o...,"['offici', 'hate', 'school', 'start', 'enter',...",offici hate school start enter caus fix,"['offici', 'hate', 'school', 'start', 'enter',..."
3,but onc again depress love to take that from m...,1,but onc again depress love to take that from m...,"['onc', 'depress', 'love', 'take', 'angri', 'w...",onc depress love take angri want cri wont come...,"[tire, angri, damn, cri, depress]"
4,"Starting today, I'm going to attempt to fix my...",0,Starting going to attempt to fix my sleep sche...,"['start', 'go', 'attempt', 'fix', 'sleep', 'sc...",start go attempt fix sleep schedul school coup...,"['start', 'go', 'attempt', 'fix', 'sleep', 'sc..."
